# 04 — AI Search smoke test

1. Create or update the chunk index
2. Insert two test chunks (with embeddings)
3. Run a hybrid query
4. Clean them up

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.search_client import SearchService
from src.openai_client import OpenAIService
from src.models import ChunkRecord

search = SearchService()
search.create_or_update_index()
ai = OpenAIService()

INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')?api-version=2024-05-01-preview'
Request method: 'PUT'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '1312'
    'api-key': 'REDACTED'
    'Prefer': 'REDACTED'
    'Accept': 'application/json;odata.metadata=minimal'
    'x-ms-client-request-id': 'c6b23755-4874-11f1-8978-df1619e59047'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=minimal; odata.streaming=true; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'Prefe

In [2]:
texts = [
    ('chunk-test-1', 'The Eiffel Tower is in Paris and is 330 meters tall.'),
    ('chunk-test-2', 'The Statue of Liberty stands in New York Harbor.'),
]
vectors = ai.embed([t for _, t in texts])
chunks = [
    ChunkRecord(id=cid, doc_id='test-doc', page=1, type='text', content=t, embedding=v)
    for (cid, t), v in zip(texts, vectors)
]
search.index_chunks(chunks)

INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.index?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '69216'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': 'cf2e6469-4874-11f1-83f6-df1619e59047'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
 

2

In [3]:
q = 'How tall is the Eiffel Tower?'
qv = ai.embed(q)[0]
for s in search.search(q, qv, top_k=2):
    print(f'{s.chunk_id}  page={s.page}  → {s.snippet[:120]}')

INFO: HTTP Request: POST https://azr-oai-dai-sandbox4-001.openai.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-08-01-preview "HTTP/1.1 200 OK"
INFO: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '34584'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': 'e193fa97-4874-11f1-a4a3-df1619e59047'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'
    'Strict-Transport-Security': 'REDAC

chunk-test-1  page=1  → The Eiffel Tower is in Paris and is 330 meters tall.
bd2759f7-91a1-4f3c-82d9-1361e709ded7  page=11  → . Version Control: The plan, todos, findings, and report drafts are all versioned within the state,
allowing full tracea


In [4]:
# Cleanup
search.delete_document('test-doc')
print('Removed test chunks ✔')

INFO: Request URL: 'https://agent-ai-search5yc2.search.windows.net/indexes('pdg-was-multimodal-rag-2')/docs/search.post.search?api-version=2024-05-01-preview'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '78'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': 'e56e87ad-4874-11f1-a90b-df1619e59047'
    'User-Agent': 'azsdk-python-search-documents/11.6.0b4 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
A body is sent with the request
INFO: Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'application/json; odata.metadata=none; odata.streaming=true; charset=utf-8'
    'Content-Encoding': 'REDACTED'
    'Vary': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'Preference-Applied': 'REDACTED'
    'OData-Version': 'REDACTED'
    'request-id': 'e56e87ad-4874-11f1-a90b-df1619e59047'
    'elapsed-time': 'REDACTED'
    'Date': 'Tue, 

Removed test chunks ✔
